# Hypothesis testing problems

## Exercise 1

You are a nutritionist researching two different types of diets to see if there is a significant difference in weight loss after one month. You choose two random groups of people; one group follows the first diet, and the other follows the second. At the end of the month, you record the weight loss (in kg) for each person:

| Diet 1 | Diet 2 |
|:-------|:-------|
| 2.0 | 3.0 |
| 2.5 | 3.2 |
| 3.0 | 3.1 |
| 2.8 | 2.9 |
| 2.3 | 2.8 |
| 2.7 | 3.0 |
| 2.5 | 3.2 |

With these data, it seeks to answer the following question: Is there a significant difference in average weight loss between people who followed the first diet and those who followed the second diet?

To draw conclusions, follow the points below:

- State the hypothesis: null and alternative hypothesis.
- Perform the test to test the hypothesis. You can use a Student's t-test.
- Analyze the conclusions.

In [1]:
import numpy as np
from scipy import stats

# 1) Data
diet1 = np.array([2.0, 2.5, 3.0, 2.8, 2.3, 2.7, 2.5])
diet2 = np.array([3.0, 3.2, 3.1, 2.9, 2.8, 3.0, 3.2])

# 2) Hypotheses
# H0: mean(diet1) == mean(diet2)
# H1: mean(diet1) != mean(diet2)   (two-sided test)

# 3) Choose significance level
alpha = 0.05

# 4) Run a two-sample t-test
# Welch t-test is safer when variances may differ
t_stat, p_value = stats.ttest_ind(diet1, diet2, equal_var=False)

print("Diet 1 mean:", diet1.mean())
print("Diet 2 mean:", diet2.mean())
print("t-statistic:", t_stat)
print("p-value:", p_value)

# 5) Decision
if p_value < alpha:
    print("Decision: reject H0 (significant difference).")
else:
    print("Decision: fail to reject H0 (no significant evidence of difference).")

Diet 1 mean: 2.542857142857143
Diet 2 mean: 3.0285714285714285
t-statistic: -3.5383407969933938
p-value: 0.007125697852423994
Decision: reject H0 (significant difference).


## ANOVA

**ANOVA** (*Analysis of Variance*) is a statistical technique used to compare the measures of two or more groups. The idea behind ANOVA is to decompose the total variability in the data into two components: between-group variability and within-group variability:

- **Between-group variability**: This variability refers to the differences between the group means. If this variability is considerably larger than the within-group variability, it could be an indication that at least one of the group means is different.
- **Within-group variability**: This variability refers to the dispersion of the data within each group. If all groups have similar variability, then any noticeable difference in group means could be considered significant.

Hypotheses in ANOVA typically include:

- **Null hypothesis** ($H₀$): The means of all groups are equal.
- **Alternative hypothesis** ($H₁$): At least one of the group means is different.

If the ANOVA test result is significant (e.g., a p-value less than a threshold such as 0.05), this suggests that at least one group mean is different.

## Exercise 2

A farmer decides to test three different types of fertilizers to determine if one is superior in terms of corn production. The farmer plants corn on 15 identical plots and uses all three fertilizers (5 plots for each type). At the end of the season, he measures the corn yield (in kg) of each plot, with the following result:

| Fertilizer 1 | Fertilizer 2 | Fertilizer 3 |
|:-------------|:-------------|:-------------|
| 20 | 22 | 24 |
| 21 | 21 | 23 |
| 20 | 23 | 22 |
| 19 | 22 | 23 |
| 20 | 21 | 24 |

With this data, he seeks to answer the following question: Is there a significant difference in average corn yield between the three types of fertilizers?

To help you, follow the points below:

- State the hypothesis: null and alternative hypothesis.
- Perform the ANOVA test.
- Analyze the conclusions.
- If one fertilizer is better than another, how can we know it?

In [6]:
# Exercise 2 – One-way ANOVA

import numpy as np
from scipy import stats
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1) Data
fert1 = np.array([20, 21, 20, 19, 20])
fert2 = np.array([22, 21, 23, 22, 21])
fert3 = np.array([24, 23, 22, 23, 24])

# 2) Hypotheses
# H0 (null hypothesis):
# The mean yield of the three fertilizers is equal.
# mu1 = mu2 = mu3

# H1 (alternative hypothesis):
# At least one fertilizer has a different mean yield.

# 3) Significance level
alpha = 0.05

# 4) Perform one-way ANOVA
F_stat, p_value = stats.f_oneway(fert1, fert2, fert3)

print("F-statistic:", F_stat)
print("p-value:", p_value)

# 5) Decision rule
if p_value < alpha:
    print("Reject H0: There is a significant difference between fertilizers.")
else:
    print("Fail to reject H0: No significant difference detected.")

#Tukey
# 1) Prepare data in long format
df = pd.DataFrame({
    "yield": np.concatenate([fert1, fert2, fert3]),
    "fertilizer": (["F1"] * len(fert1)) +
                  (["F2"] * len(fert2)) +
                  (["F3"] * len(fert3))
})

print(df)

# 2) Run Tukey HSD
tukey = pairwise_tukeyhsd(
    endog=df["yield"],        # numerical variable
    groups=df["fertilizer"],  # group labels
    alpha=0.05
)

print(tukey)


F-statistic: 20.31578947368421
p-value: 0.00014047824793190402
Reject H0: There is a significant difference between fertilizers.
    yield fertilizer
0      20         F1
1      21         F1
2      20         F1
3      19         F1
4      20         F1
5      22         F2
6      21         F2
7      23         F2
8      22         F2
9      21         F2
10     24         F3
11     23         F3
12     22         F3
13     23         F3
14     24         F3
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower  upper  reject
--------------------------------------------------
    F1     F2      1.8 0.0099 0.4572 3.1428   True
    F1     F3      3.2 0.0001 1.8572 4.5428   True
    F2     F3      1.4 0.0409 0.0572 2.7428   True
--------------------------------------------------


### Tukey HSD Results

- **F1 vs F2** → significant  
- **F1 vs F3** → significant  
- **F2 vs F3** → significant  

### Conclusion

$F3 > F2 > F1$

All differences between fertilizers are **statistically significant**.